In [23]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
import cv2
from tqdm import tqdm
from pykalman import KalmanFilter
from tqdm import tqdm

# Kalman Filter Object

In [3]:
class Kalman(object):
    def __init__(self, F, Q, H, R, u):
        """
        Initialize the dynamical system models.

        Parameters
        ----------
        F : ndarray of shape (n,n)
            The state transition model.
        Q : ndarray of shape (n,n)
            The covariance matrix for the state noise.
        H : ndarray of shape (m,n)
            The observation model.
        R : ndarray of shape (m,m)
            The covariance matrix for observation noise.
        u : ndarray of shape (n,)
            The control vector.
        """
        # Initialize given parameters
        self.F = F
        self.Q = Q
        self.H = H
        self.R = R
        self.u = u
   

    def estimate(self, x, P, z):
        """
        Compute the state estimates using the Kalman filter.
        If x and P correspond to time step k, then z is a sequence of
        observations starting at time step k+1.

        Parameters
        ----------
        x : ndarray of shape (n,)
            The initial state estimate.
        P : ndarray of shape (n,n)
            The initial error covariance matrix.
        z : ndarray of shape (m,N)
            Sequence of N observations (each column is an observation).

        Returns
        -------
        out : ndarray of shape (n,N)
            Sequence of state estimates (each column is an estimate).
        Ps : ndarray of shape (n,n,N)
            Sequence of smoothed covariance estimates.
        """
        n = len(x)
        N = z.shape[1]

        # Store initial state estimates
        out = np.zeros((n, N))
        Ps = np.zeros((n, n, N))
        Ps_predicted = np.zeros((n, n, N))
        x0, P0 = x, P
        out[:, 0] = x0
        Ps[:, :, 0] = P0

        for i in range(1, N):
            # Predict steps for xk and Pk
            xk = self.F @ x0 + self.u
            Pk = self.F @ P0 @ self.F.T + self.Q
            Pk = (Pk + Pk.T) / 2
            Ps_predicted[:, :, i] = Pk

            # Update steps for Kk, xk, and Pk
            Kk = np.linalg.solve((self.H @ Pk @ self.H.T + self.R).T,
                                 (Pk @ self.H.T).T).T
            xk = xk + Kk @ (z[:, i] - self.H @ xk)
            Pk = (np.eye(n) - Kk @ self.H) @ Pk
            Pk = (Pk + Pk.T) / 2

            out[:, i] = xk
            Ps[:, :, i] = Pk
            x0, P0 = xk, Pk

        return out, Ps, Ps_predicted
            
    
    def predict(self, x, k):
        """
        Predict the next k states in the absence of observations.

        Parameters
        ----------
        x : ndarray of shape (n,)
            The current state estimate.
        k : integer
            The number of states to predict.

        Returns
        -------
        out : ndarray of shape (n,k)
            The next k predicted states.
        """
        # Initialize array to store predicted states
        n = len(x)
        out = np.zeros((n, k))
        x0 = x
        out[:, 0] = x0

        for i in range(1, k):
            # Perform update step without observations
            xk = np.dot(self.F, x0) + self.u
            out[:, i] = xk
            x0 = xk

        return out
    

    def rewind(self, x, P, Ps_filtered, Ps_predicted):
        """
        Predict the k states and smoothed covariances
        preceding the current state estimate x.

        Parameters
        ----------
        x : ndarray of shape (n,)
            The current state estimate.
        P : ndarray of shape (n,n)
            The current smoothed covariance estimate.
        T : integer
            The number of preceding states to predict.

        Returns
        -------
        out : ndarray of shape (n,k)
            The k preceding predicted states.
        Ps : ndarray of shape (n,n,k)
            The k preceding smoothed covariances.
        """
        # Initialize prediction arrays
        T = Ps_filtered.shape[2]
        n = len(x)
        out = np.zeros((n, T))
        Ps = np.zeros((n, n, T))

        # Store known state as last column
        out[:, -1] = x
        Ps[:, :, -1] = P
        
        for i in range(T-2, -1, -1):
            # Predicted covariance one step aghead
            P_filt = Ps_filtered[:, :, i]
            P_pred = Ps_predicted[:, :, i+1]
            P_pred = (P_pred + P_pred.T) / 2

            # RTS Smoother gain
            Gk = np.linalg.solve(P_pred.T, (P_filt @ self.F.T).T).T

            out[:, i] = out[:, i+1]
            out[:, i] = Ps_filtered[:, :, i] @ np.zeros(n)

            x_filt = out[:, i+1]
            Pk = P_filt + Gk @ (Ps[:, :, i+1] - P_pred) @ Gk.T
            Ps[:, :, i] = (Pk + Pk.T) / 2

        return out, Ps
    
    def smooth(self, x, P, z):
        """Combines forward and backward pass."""
        n = len(x)
        T = z.shape[1]

        # Forward pass
        xs = np.zeros((n, T))
        Ps_filt = np.zeros((n, n, T))
        Ps_pred = np.zeros((n, n, T))

        x0, P0 = x, P
        xs[:, 0] = x0
        Ps_filt[:, :, 0] = P0
        Ps_pred[:, :, 0] = P0

        for i in range(1, T):
            xk = self.F @ x0 + self.u
            Pk = self.F @ P0 @ self.F.T + self.Q
            Pk = (Pk + Pk.T) / 2
            Ps_pred[:, :, i] = Pk

            Kk = np.linalg.solve((self.H @ Pk @ self.H.T + self.R).T,
                                (Pk @ self.H.T).T).T
            xk = xk + Kk @ (z[:, i] - self.H @ xk)
            Pk = (np.eye(n) - Kk @ self.H) @ Pk
            Pk = (Pk + Pk.T) / 2

            xs[:, i] = xk
            Ps_filt[:, :, i] = Pk
            x0, P0 = xk, Pk

        # Backward pass
        x_smooth = xs.copy()
        Ps_smooth = Ps_filt.copy()

        for i in range(T-2, -1, -1):
            P_pred = Ps_pred[:, :, i+1]
            P_pred = (P_pred + P_pred.T) / 2

            Gk = np.linalg.solve(P_pred.T, (Ps_filt[:, :, i] @ self.F.T).T).T

            x_smooth[:, i] = xs[:, i] + Gk @ (x_smooth[:, i+1] - self.F @ xs[:, i] - self.u)
            Pk = Ps_filt[:, :, i] + Gk @ (Ps_smooth[:, :, i+1] - P_pred) @ Gk.T
            Ps_smooth[:, :, i] = (Pk + Pk.T) / 2

        return x_smooth, Ps_smooth, Ps_filt

# Known parameters for Kalman Filter

In [4]:
# State transiiton matrix 
# Follows forward Euler x_{k+1} = x_k + dx
F = np.array([
    [1, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 0, 0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0, 0, 1, 0],
    [0, 0, 0, 1, 0, 0, 0, 1],
    [0, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 1]
])

# First four states are observed at each step
H = np.array([
    [1, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0]
])

n = F.shape[0]  # Number of predicted states
m = H.shape[0]  # Number of observed states

# We don't use control for this
u = np.zeros(n)

# Create test/training splits from category 3 data

In [19]:
# Category 3 videos include people moving around in the screen
# video_files = ['410', '411', '516', '517', '526', '528', '529', '530', '531', '533', 
#             #    '557',
#                '558', '559', '562']
video_files = [video.name for video in os.scandir('data') if video.is_dir()]

X_train_temp, X_test = train_test_split(video_files, test_size=0.15, random_state=42)
X_train, X_val = train_test_split(X_train_temp, test_size=0.1765, random_state=42)

# Helper functions for processing video data

In [38]:
def get_bounding_box(filename):
    with open(filename) as f:
        data = f.readlines()

    combined = ' '.join(data)
    start = combined.index("{") + 3
    end = combined.index("}")
    points = [coord.split(" ") for coord in combined[start:end].strip().split("\n ")]
    points = np.array(points).astype(float)

    max_coords = np.max(points, axis=0)
    min_coords = np.min(points, axis=0)
    x, y = min_coords
    w, h = max_coords - min_coords
    
    return x, y, w, h

def get_true_states(video_id):
    filename = f"data/{video_id}/annot"
    frames = sorted([file for file in os.listdir(filename) if file.endswith(".pts")])
    Z = []

    for frame in frames:
        file = filename + "/" + frame
        x, y, w, h = get_bounding_box(file)
        Z.append([x, y, w, h])

    return np.array(Z).T

def get_observations(video_id, force_recompute=False):
    cache_path = f"data/{video_id}/observations.npy"
    none_mask_path = f"data/{video_id}/none_mask.npy"

    # Return cached result if it exists
    if os.path.exists(cache_path) and not force_recompute:
        Z = np.load(cache_path)
        none_mask = np.load(none_mask_path)
        return Z, none_mask

    # Load detector
    detector = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    cap = cv2.VideoCapture(f"data/{video_id}/vid.avi")

    observations = []
    none_mask = []

    while True:
        ret, frame = cap.read()
        if not ret or frame is None or frame.size == 0:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        detections = detector.detectMultiScale(gray,
                                               scaleFactor=1.1,
                                               minNeighbors=5,
                                               minSize=(60, 60))

        if len(detections) > 0:
            x, y, w, h = detections[0]
            observations.append([x, y, w, h])
            none_mask.append(False)
        else:
            observations.append([0, 0, 0, 0])
            none_mask.append(True)

    cap.release()

    Z = np.array(observations).T
    none_mask = np.array(none_mask)

    np.save(cache_path, Z)
    np.save(none_mask_path, none_mask)
    return Z, none_mask

def fill_observations(Z, none_mask, F, H, Q, R, n, m):
    T = Z.shape[1]
    Z_filled = Z.copy()

    # Wait for first real detection to initialize
    first = np.argmax(~none_mask)
    x_hat = np.concatenate((Z[:, first], np.zeros(m)))
    P = np.diag([100, 100, 100, 100, 10, 10, 10, 10]).astype(float)

    for i in range(first, T):
        # Predict
        x_hat = F @ x_hat
        P = F @ P @ F.T + Q

        if none_mask[i]:
            z_t = H @ x_hat  # Use prediction as fallback
            Z_filled[:, i] = z_t
        else:
            z_t = Z[:, i]

        # Update
        K = np.linalg.solve((H @ P @ H.T @ R).T, (P @ H.T).T).T
        x_hat = x_hat + K @ (z_t - H @ x_hat)
        P = (np.eye(n) - K @ H) @ P
        P = (P + P.T) / 2

    return Z_filled

def get_continuous_chunks(video_id):
    Z, none_mask = get_observations(video_id)

    split_indices = np.where(none_mask)[0]
    segments = np.split(Z.T, split_indices)
    cleaned_segments = [segments[0]] + [seg[1:] for seg in segments[1:]]

    MIN_LENGTH = 10
    return [seg for seg in cleaned_segments if seg.shape[0] >= MIN_LENGTH]

# Use OpenCV to generate observed face states at each frame in video data

In [60]:
video_files = [video.name for video in os.scandir('data') if video.is_dir()]

for video_id in tqdm(video_files):
    get_observations(video_id, force_recompute=True)

100%|██████████| 114/114 [1:01:41<00:00, 32.47s/it]


# EM-estimation using guesses for Q and R

In [ ]:
# Initial guesses for noise covariances
Q = 0.1 * np.eye(n)
R = 500 * np.eye(m)

n_em_iters = 200

for em_iter in range(n_em_iters):
    # Instantiate a Kalman filter
    kf = Kalman(F, Q, H, R, u)

    # Accumulated estimates across all training videos
    R_accum = np.zeros((m, m))
    Q_accum = np.zeros((n, n))
    total_frames = 0

    for video_id in X_train:
        # Get cached observations
        Z_raw, none_mask = get_observations(video_id)

        # Fill in non-detected frames with Kalman update
        Z = fill_observations(Z_raw, none_mask, F, H, Q, R, n, m)
        T = Z.shape[1]

        first = np.argmax(~none_mask)  # Index of first valid detection

        # Expectation step: Forward Kalman filter
        z0 = Z[:, first] 
        x0 = np.concatenate((z0, np.zeros(m)))  # Guess zero initial velocity
        P0 = np.diag([100, 100, 100, 100, 10, 10, 10, 10]).astype(float)
        
        x_smooth, Ps_smooth, Ps_filt = kf.smooth(x0, P0, Z[:, first:])

        residuals = Z[:, first:] - H @ x_smooth
        if np.any(np.diag(residuals) > 1e8):
            print(f"R blowing up on video {video_id}, skipping video")
            continue

        T_valid = T - first  # Number of frames actually used

        # Maximization step: Accumulate R and Q updates
        for i in range(T_valid):
            residual = residuals[:, i]
            R_accum += np.outer(residual, residual) + H @ Ps_smooth[:, :, i] @ H.T

        for i in range(1, T_valid):
            P_pred = F @ Ps_smooth[:, :, i-1] @ F.T + Q
            P_pred = (P_pred + P_pred.T) / 2
            Gk = np.linalg.solve(P_pred.T, (Ps_smooth[:, :, i-1] @ F.T).T).T
            P_cross = Ps_smooth[:, :, i] @ Gk.T

            Q_accum += (Ps_smooth[:, :, i]
                        - F @ P_cross.T
                        - P_cross @ F.T
                        + F @ Ps_smooth[:, :, i-1] @ F.T)
            
        total_frames += T_valid

    # Noramlize by total frames across all training videos
    R = R_accum / total_frames
    R = (R + R.T) / 2
    R = np.maximum(R, 1e-4 * np.eye(m))

    Q = Q_accum / total_frames
    Q = (Q + Q.T) / 2
    Q = np.maximum(Q, 1e-4 * np.eye(n))

    print(f"EM iter {em_iter+1}/{n_em_iters} complete - "
          f"diag(R): {np.diag(R).round(2)}, diag(Q): {np.diag(Q).round(4)}")

EM iter 1/200 complete - diag(R): [1885.65 1948.48  749.73  749.73], diag(Q): [0.2041 0.2041 0.2041 0.2041 0.1509 0.1509 0.1509 0.1509]
EM iter 2/200 complete - diag(R): [2021.58 1997.15  781.07  781.07], diag(Q): [1.5738 1.5877 0.935  0.935  0.2386 0.239  0.2008 0.2008]
EM iter 3/200 complete - diag(R): [1975.22 1952.65  765.59  765.59], diag(Q): [5.9893 6.0069 3.2068 3.2068 0.3659 0.3661 0.2322 0.2322]


/tmp/ipykernel_22512/1804353446.py:91: RuntimeWarning: invalid value encountered in cast
  Z_filled[:, i] = z_t
/tmp/ipykernel_22512/1804353446.py:96: RuntimeWarning: overflow encountered in matmul
  K = np.linalg.solve((H @ P @ H.T @ R).T, (P @ H.T).T).T


EM iter 4/200 complete - diag(R): [1.90644000e+03 1.89272000e+03 1.28183648e+36 1.28197847e+36], diag(Q): [14.9132 14.8949  6.7298  6.7298  0.5348  0.5345  0.2757  0.2757]
EM iter 5/200 complete - diag(R): [1.83035000e+03 1.82447000e+03 2.97050671e+08 2.97050583e+08], diag(Q): [27.5575 27.4254  6.7263  6.7263  0.7307  0.7295  0.2755  0.2755]
EM iter 6/200 complete - diag(R): [  1760.5    1760.9  588613.79 588613.62], diag(Q): [ 24.1583  23.3015 189.0193 189.0193   0.8472   0.8423   0.5094   0.5094]
EM iter 7/200 complete - diag(R): [1762.34 1770.63 8709.63 8709.62], diag(Q): [ 77.7002  75.9215 757.9447 757.9446   1.2167   1.2089   0.934    0.934 ]
EM iter 8/200 complete - diag(R): [1624.55 1633.76 1761.04 1761.04], diag(Q): [1.397801e+02 1.393300e+02 9.905363e+02 9.905361e+02 1.547700e+00
 1.541000e+00 9.579000e-01 9.579000e-01]
EM iter 9/200 complete - diag(R): [1516.12 1526.99  897.59  897.59], diag(Q): [2.025919e+02 2.022578e+02 7.685270e+02 7.685269e+02 1.771800e+00
 1.765600e+00 4

/tmp/ipykernel_22512/1804353446.py:97: RuntimeWarning: overflow encountered in matmul
  x_hat = x_hat + K @ (z_t - H @ x_hat)
/tmp/ipykernel_22512/1804353446.py:97: RuntimeWarning: invalid value encountered in matmul
  x_hat = x_hat + K @ (z_t - H @ x_hat)
/tmp/ipykernel_22512/1804353446.py:98: RuntimeWarning: overflow encountered in matmul
  P = (np.eye(n) - K @ H) @ P
/tmp/ipykernel_22512/1804353446.py:87: RuntimeWarning: invalid value encountered in matmul
  P = F @ P @ F.T + Q


R blowing up on video 044, skipping video
EM iter 18/200 complete - diag(R): [6.20310995e+35 6.21800951e+35 6.66920525e+35 6.66919977e+35], diag(Q): [3.462071e+02 3.491819e+02 1.306685e+02 1.306682e+02 8.121000e-01
 8.148000e-01 1.633000e-01 1.632000e-01]
EM iter 19/200 complete - diag(R): [8.65982187e+08 8.68967590e+08 1.85545333e+08 1.85542062e+08], diag(Q): [3.460253e+02 3.489986e+02 1.305999e+02 1.305995e+02 8.116000e-01
 8.144000e-01 1.632000e-01 1.632000e-01]
EM iter 20/200 complete - diag(R): [1738122.25 1743315.29  373671.66  373665.11], diag(Q): [2.6810470e+03 2.6764482e+03 5.5360390e+02 5.5360050e+02 1.5174000e+00
 1.4999000e+00 3.0410000e-01 3.0410000e-01]
EM iter 21/200 complete - diag(R): [38815.22 38730.48  8118.84  8118.75], diag(Q): [6.1627045e+03 6.1516305e+03 1.2589201e+03 1.2589095e+03 2.7131000e+00
 2.6814000e+00 5.5660000e-01 5.5650000e-01]
EM iter 22/200 complete - diag(R): [8622.43 8607.29 1958.11 1958.1 ], diag(Q): [6.9204605e+03 6.9067375e+03 1.4044276e+03 1.40

# Store noise covariances

In [ ]:
np.save('parameters/R.npy', R)
np.save('parameters/Q.npy', Q)
print("Saved R and Q")

# Using `pykalman`

In [ ]:
all_segments = []
for video_id in X_train:
    chunks = get_continuous_chunks(video_id)
    all_segments.extend(chunks)

kf = KalmanFilter(
    transition_matrices=F,
    observation_matrices=H,
    transition_covariance=0.1 * np.eye(8),
    observation_covariance=500 * np.eye(4),
    n_dim_state=8,
    n_dim_obs=4,
    em_vars=['transition_covariance', 'observation_covariance']
)

Q_est, R_est = None, None

for seq in tqdm(all_segments):
    kf = kf.em(
        seq,
        n_iter=5,
        em_vars=['transition_covariance', 'observation_covariance']
    )

Q_est = kf.transition_covariance
R_est = kf.observation_covariance

print("Estimated Q:", np.diag(Q_est))
print("Estimated R:", np.diag(R_est))

 10%|█         | 63/606 [00:19<00:21, 25.20it/s]/home/connor-mcbride/acme/math404/final-project/em-kalman-face-tracker/.venv/lib/python3.12/site-packages/pykalman/standard.py:477: RuntimeWarning: overflow encountered in dot
  smoothed_state_covariance = filtered_state_covariance + np.dot(
/home/connor-mcbride/acme/math404/final-project/em-kalman-face-tracker/.venv/lib/python3.12/site-packages/pykalman/standard.py:479: RuntimeWarning: invalid value encountered in dot
  np.dot(
/home/connor-mcbride/acme/math404/final-project/em-kalman-face-tracker/.venv/lib/python3.12/site-packages/pykalman/standard.py:578: RuntimeWarning: invalid value encountered in dot
  pairwise_covariances[t] = np.dot(
/home/connor-mcbride/acme/math404/final-project/em-kalman-face-tracker/.venv/lib/python3.12/site-packages/pykalman/standard.py:789: RuntimeWarning: invalid value encountered in dot
  np.dot(smoothed_state_covariances[t], observation_matrix.T),
/home/connor-mcbride/acme/math404/final-project/em-kalman-

ValueError: array must not contain infs or NaNs

In [ ]:
Q_accum = np.zeros((8, 8))
R_accum = np.zeros((4, 4))
total_frames = 0

all_segments = []
for video_id in X_train:
    chunks = get_continuous_chunks(video_id)
    all_segments.extend(chunks)

for seq in tqdm(all_segments):
    T = seq.shape[0]

    kf_seq = KalmanFilter(
        transition_matrices=F,
        observation_matrices=H,
        transition_covariance=0.1 * np.eye(8),
        observation_covariance=500 * np.eye(4),
        initial_state_mean=np.concatenate((seq[0], np.zeros(4))),
        initial_state_covariance=np.diag([100, 100, 100, 100, 10, 10, 10, 10]),
        n_dim_state=8,
        n_dim_obs=4
    )

    try:
        kf_seq = kf_seq.em(seq, n_iter=10,
                           em_vars=['transition_covariance', 'observation_covariance'])
    except ValueError:
        continue  # skip segments that cause numerical issues

    # Weight contribution by segment length
    Q_accum += T * kf_seq.transition_covariance
    R_accum += T * kf_seq.observation_covariance
    total_frames += T

Q_est = Q_accum / total_frames
R_est = R_accum / total_frames

print("Estimated Q diag:", np.diag(Q_est).round(4))
print("Estimated R diag:", np.diag(R_est).round(4))

100%|██████████| 606/606 [03:50<00:00,  2.63it/s]

Estimated Q diag: [0.1019 0.102  0.0575 0.0575 0.2959 0.1888 0.0847 0.0847]
Estimated R diag: [1907.8952 1958.1056  886.5147  886.5139]


In [50]:
np.save('parameters/R.npy', R_est)
np.save('parameters/Q.npy', Q_est)
print("Saved R and Q")

Saved R and Q
